# CIC-IDS2017: перевірка таймстемпів (AM/PM) і складу атак у temporal split 70/85
Запускайте по клітинках у Jupyter (або весь файл як скрипт).
Нічого не змінює у ваших файлах — лише читає і друкує.

In [1]:
# 0. Автопошук файлу на сервері (на випадок, якщо реальна назва відрізняється)
import subprocess

result = subprocess.run(
    ["find", "/srv/ids_research", "/home/ivan_r_opirskyy", "-iname", "*cic_ids2017*", "-type", "f"],
    capture_output=True, text=True, timeout=60
)
print(result.stdout)
print(result.stderr)

/srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet
/srv/ids_research/data/cic_ids2017_flows.parquet
/srv/ids_research/data/cic_ids2017_flows_fixed.parquet




In [2]:
# 1. Налаштування — ВКАЖІТЬ ШЛЯХ до того самого файлу, з якого будувався спліт
import pandas as pd, numpy as np

# Підтверджений шлях (re-extracted через пакет nids-datasets, зафіксований у проєкті):
PATH = "/srv/ids_research/data/cic_ids2017_flows_fixed.parquet"
# Якщо клітинка 0 вище знайшла файл із 'cve_context'/'enriched' у назві (саме з нього
# будувався F0/F1 спліт у статті) — підставте той шлях сюди замість цього.

TS_COL = "Timestamp"
LABEL_CANDIDATES = ["attack_label", "Label", "label", "attack_cat"]

df = pd.read_parquet(PATH) if PATH.endswith(".parquet") else pd.read_csv(PATH, low_memory=False)
df.columns = [c.strip() for c in df.columns]
LABEL_COL = next(c for c in LABEL_CANDIDATES if c in df.columns)
print("rows:", len(df), "| label column:", LABEL_COL)
print(df[LABEL_COL].value_counts().to_string())

rows: 2827677 | label column: attack_label
attack_label
BENIGN                        2271122
DoS Hulk                       230123
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack – Brute Force         1507
Web Attack – XSS                  652
Infiltration                       36
Web Attack – Sql Injection         21
Heartbleed                         11


In [3]:
# 2. Як виглядають СИРІ таймстемпи
raw = df[TS_COL]
print("dtype:", raw.dtype)
print(raw.astype(str).sample(15, random_state=0).to_string())
s = raw.astype(str)
print("\nмістять 'AM'/'PM':", s.str.contains(r"\b[AP]M\b", case=False).sum())

dtype: datetime64[ns]
928097     2017-07-04 01:28:00
1026445    2017-07-05 10:17:00
1985440    2017-07-06 09:33:00
854040     2017-07-04 02:44:00
514231     2017-07-03 01:37:02
1610936    2017-07-05 05:01:00
2379460    2017-07-07 02:24:00
1201217    2017-07-05 10:53:00
2737802    2017-07-07 04:09:00
2326233    2017-07-07 01:25:00
2663884    2017-07-07 04:01:00
1338559    2017-07-05 12:47:00
1182750    2017-07-05 10:52:00
2694143    2017-07-07 04:04:00
2167964    2017-07-07 09:54:00

містять 'AM'/'PM': 0


In [4]:
# 3. Парсинг так само, як у статті (day-first), і розподіл годин по днях
ts = raw if np.issubdtype(raw.dtype, np.datetime64) else pd.to_datetime(s, dayfirst=True, errors="coerce")
print("не розпарсилось:", ts.isna().sum())
df["_ts"] = ts
df["_day"] = ts.dt.date
df["_hour"] = ts.dt.hour
print("\nКількість потоків: день × година")
print(pd.crosstab(df["_day"], df["_hour"]).to_string())
# Захоплення CIC-IDS2017 йшло приблизно 08:00–17:00. Багато записів з годинами 1–5
# і мало з 13–17 => 12-годинний формат без AM/PM (pm-час записано як 1–5).

не розпарсилось: 0

Кількість потоків: день × година
_hour          1       2       3       4     5      8      9       10     11     12
_day                                                                               
2017-07-03  57561   48532   48894   65829  1840    772  74341   85008  88781  57892
2017-07-04  36779   72614   48111   62969   165   1677  89635   58487  57138  18066
2017-07-05  48867   63799   66094   44853  3531  14095  54263  300911  71002  23975
2017-07-06  50087   57665  129228   48807  2466      6  62252   45785  42659  19528
2017-07-07  43020  191296  104265  171671  1552      2  49935   71535  55394  14043


In [5]:
# 4. Час кожної атаки vs офіційний розклад (Sharafaldin et al., 2018)
expected = {
    "FTP-Patator": "Tue 09:20–10:20", "SSH-Patator": "Tue 14:00–15:00",
    "DoS slowloris": "Wed 09:47–10:10", "DoS Slowhttptest": "Wed 10:14–10:35",
    "DoS Hulk": "Wed 10:43–11:00", "DoS GoldenEye": "Wed 11:10–11:23",
    "Heartbleed": "Wed 15:12–15:32", "Web Attack": "Thu 09:20–10:42",
    "Infiltration": "Thu 14:19–15:45", "Bot": "Fri 10:02–11:02",
    "PortScan": "Fri 13:55–14:35", "DDoS": "Fri 15:56–16:16",
}
g = df.groupby(LABEL_COL)["_ts"].agg(["min", "max", "count"])
g["expected"] = [next((v for k, v in expected.items() if k.lower() in str(i).lower()), "") for i in g.index]
print(g.to_string())
# Якщо PortScan/DDoS/SSH-Patator/Infiltration показують години ~01–04 замість 13–16 — AM/PM підтверджено.

                                           min                 max    count         expected
attack_label                                                                                
BENIGN                     2017-07-03 01:00:01 2017-07-07 12:59:00  2271122                 
Bot                        2017-07-07 09:34:00 2017-07-07 12:59:00     1956  Fri 10:02–11:02
DDoS                       2017-07-07 03:56:00 2017-07-07 04:16:00   128025  Fri 15:56–16:16
DoS GoldenEye              2017-07-05 11:10:00 2017-07-05 11:19:00    10293  Wed 11:10–11:23
DoS Hulk                   2017-07-05 10:43:00 2017-07-05 11:07:00   230123  Wed 10:43–11:00
DoS Slowhttptest           2017-07-05 10:15:00 2017-07-05 10:37:00     5499  Wed 10:14–10:35
DoS slowloris              2017-07-05 02:24:00 2017-07-05 10:11:00     5796  Wed 09:47–10:10
FTP-Patator                2017-07-04 09:17:00 2017-07-04 10:30:00     7935  Tue 09:20–10:20
Heartbleed                 2017-07-05 03:12:00 2017-07-05 03:32:00    

In [6]:
# 5. Відтворення спліту 70/85 зі статті і склад атак по партиціях
def split_by_percentile(t):
    v = np.sort(t.dropna().values)
    b1 = v[int(round(0.70 * (len(v) - 1)))]   # найближче реально наявне значення
    b2 = v[int(round(0.85 * (len(v) - 1)))]
    part = np.where(t < b1, "train", np.where(t < b2, "val", "test"))
    return pd.Series(part, index=t.index), pd.Timestamp(b1), pd.Timestamp(b2)

df["_split"], b1, b2 = split_by_percentile(df["_ts"])
print("межі:", b1, "|", b2, "  (у статті: 2017-07-06 09:28 | 2017-07-07 03:23)")
ct = pd.crosstab(df[LABEL_COL], df["_split"])[["train", "val", "test"]]
print(ct.to_string())
print("\nКласи атак, яких НЕМАЄ в train, але є у val/test:")
benign = [c for c in ct.index if str(c).lower() in ("benign", "normal", "0")]
print(ct[(ct["train"] == 0) & ((ct["val"] + ct["test"]) > 0)].drop(index=benign, errors="ignore").to_string())

межі: 2017-07-06 09:28:00 | 2017-07-07 03:23:00   (у статті: 2017-07-06 09:28 | 2017-07-07 03:23)
_split                        train     val    test
attack_label                                       
BENIGN                      1713498  262470  295154
Bot                               0       0    1956
DDoS                              0       0  128025
DoS GoldenEye                 10293       0       0
DoS Hulk                     230123       0       0
DoS Slowhttptest               5499       0       0
DoS slowloris                  5796       0       0
FTP-Patator                    7935       0       0
Heartbleed                       11       0       0
Infiltration                     36       0       0
PortScan                          0  154799    4005
SSH-Patator                    5897       0       0
Web Attack – Brute Force        258    1249       0
Web Attack – Sql Injection        0      21       0
Web Attack – XSS                  0     652       0

Класи атак, яких 

In [7]:
# 6. Що буде після виправлення AM/PM (години 1–7 => +12 год)
fixed = df["_ts"].where(df["_hour"] >= 8, df["_ts"] + pd.Timedelta(hours=12))
df["_split_fixed"], fb1, fb2 = split_by_percentile(fixed)
print("виправлені межі:", fb1, "|", fb2)
print(pd.crosstab(df[LABEL_COL], df["_split_fixed"])[["train", "val", "test"]].to_string())
# Порівняйте з п.5: чи змінюються межі і склад атак у val/test.

виправлені межі: 2017-07-06 15:14:00 | 2017-07-07 14:51:00
_split_fixed                  train     val    test
attack_label                                       
BENIGN                      1707824  425493  137805
Bot                               0    1956       0
DDoS                              0       0  128025
DoS GoldenEye                 10293       0       0
DoS Hulk                     230123       0       0
DoS Slowhttptest               5499       0       0
DoS slowloris                  5796       0       0
FTP-Patator                    7935       0       0
Heartbleed                       11       0       0
Infiltration                     22      14       0
PortScan                          0     372  158432
SSH-Patator                    5897       0       0
Web Attack – Brute Force       1507       0       0
Web Attack – Sql Injection       21       0       0
Web Attack – XSS                652       0       0
